## Prerequisites

This notebook demonstrates how to invoke a **deployed real-time inference endpoint** on Azure for the Clinical De-identification model. Follow the steps below in order before running this notebook.

### Deployment Sequence

1. **Acquire the Azure Application Offer** — Subscribe to the *clinical_deidentification_docwise_wip_en* offer in the [Azure Marketplace](https://marketplace.azure.com) to obtain the Solution Template.
2. **Launch guided deployment** — From the Marketplace listing, start the guided deployment flow for the solution.
3. **Supply configuration** — Complete the guided UX forms (defined by the offer's `createUiDefinition`) with the required parameters.
4. **Deploy resources** — Confirm deployment; Azure provisions the resources defined in the offer's `mainTemplate.json`.
5. **Wait for deployment to succeed** — Verify the deployment reaches a *Succeeded* state before proceeding. Note the endpoint name from the deployment output.
6. **Run this notebook** — Execute the cells below against the real-time inference endpoint provisioned by the deployment.

### Environment

- **Notebook runtime**: This notebook is intended to run in **Azure Machine Learning workspace notebooks** (e.g. via an Azure ML Studio compute instance or Studio notebooks).
- **Jupyter kernel**: Use a kernel with the `azure.ai.ml` Python package installed, for example **Python 3.10 — SDK v2** (`python310-sdkv2`).
- **Authentication**: The kernel must be authenticated to Azure so that Azure ML SDK calls succeed (e.g. `DefaultAzureCredential()`). The authenticated identity must be granted Azure RBAC permissions appropriate for reading the workspace and online endpoint configuration, and for retrieving endpoint credentials. Consult your organization's access-policy guidance and the [Azure ML endpoint credential documentation](https://learn.microsoft.com/azure/machine-learning/how-to-manage-endpoints) for details.
- **Network access**: The notebook's compute instance must be able to reach the endpoint's `scoring_uri` (no outbound firewall or VNET restrictions that block it).


## Clinical Deidentification Arabic

Deidentification is essential for safeguarding patient privacy in clinical data, including texts, PDFs, images, and DICOM files containing Protected Health Information (PHI). PHI encompasses various health-related data, including common identifiers such as name, address, birth date, and Social Security Number.

- **Model**: [Arabic](https://nlp.johnsnowlabs.com/2021/03/15/clinical_deidentification_ar.html)
- **Model Description**: This pipeline can be used to deidentify Arabic PHI information from medical texts. The PHI information will be masked and obfuscated in the resulting text. The pipeline can mask and obfuscate CONTACT, NAME, DATE, ID, LOCATION, AGE, PATIENT, HOSPITAL, ORGANIZATION, CITY, STREET, USERNAME, SEX, IDNUM, EMAIL, ZIP, MEDICALRECORD, PROFESSION, PHONE, COUNTRY, DOCTOR, SSN, ACCOUNT, LICENSE, DLN and VIN

In [ ]:
ENDPOINT_NAME = "<CUSTOMER_TO_PUT_DEPLOYED_ENDPOINT_NAME>"

### Important Parameters

- **masking_policy**: `str`

    Users can select a masking policy to determine how sensitive entities are handled:

    Example: "**ملاحظات سريرية - مريض السكري: التاريخ: 30 مايو 2023، اسم المريض: فاطمة علي، العنوان: شارع الحرية، مستشفى الشفاء، د. محمد صلاح**"

    - **masked**: Default policy that masks entities with their type.

      -> 'ملاحظات سريرية - مريض `<PATIENT>`: التاريخ: `<DATE>`، اسم المريض: `<NAME>`، العنوان: `<STREET>`، `<HOSPITAL>`، `<DOCTOR>`'

    - **obfuscated**: Replaces sensitive entities with random values of the same type.

      -> 'ملاحظات سريرية - مريض `سارة أحمد`، التاريخ: `15 مارس 2022`، اسم المريض: `ليلى محمد`، العنوان: `شارع النصر`، `مستشفى الأمل`، `د. أحمد حسن`'

    - **masked_fixed_length_chars**: Masks entities with a fixed length of asterisks (\*).

      -> 'ملاحظات سريرية - مريض `****`، التاريخ: `****`، اسم المريض: `****`، العنوان: `****`، `****`، `****`'

    - **masked_with_chars**: Masks entities with asterisks (\*) matching the length of each entity.

      -> 'ملاحظات سريرية - مريض [`*********`]، التاريخ: [`***********`]، اسم المريض: [`*********`]، العنوان: [`***********`]، [`*************`]، [`************`]'
    
You can specify these parameters in the input as follows:

```json
{
    "text": [
        "Text document 1",
        "Text document 2"
    ],
    "masking_policy": "masked"
}
```

> **Note:** Demonstration inputs used with this endpoint should consist of approved synthetic or properly de-identified data. Never submit real protected health information (PHI) without explicit authorization and the required safeguarding controls.


### Authentication and Endpoint Lookup

The cell below uses `DefaultAzureCredential()` to authenticate to Azure, then retrieves the endpoint's scoring URI and primary access key from the workspace configuration. If you see authentication errors, verify that you are logged into Azure Cloud Shell / your compute instance and that your identity has sufficient role assignments on the workspace.


In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import requests
import json


ml_client = MLClient.from_config(DefaultAzureCredential())
endpoint_url = ml_client.online_endpoints.get(ENDPOINT_NAME).scoring_uri
key = ml_client.online_endpoints.get_keys(ENDPOINT_NAME).primary_key


def invoke_realtime_endpoint(record, content_type="application/json", accept="application/json"):
    response = requests.post(
        endpoint_url,
        headers={
            "Authorization": f"Bearer {key}",
            "Content-Type": content_type,
        },
        data=json.dumps(record, ensure_ascii=False) if content_type == "application/json" else record,
     
    )

    response_body = response.content.decode("utf-8")

    if accept == "application/json":
        return json.loads(response_body)
    elif accept == "application/jsonlines":
        return [json.loads(line) for line in response_body.strip().split('\n') if line]
    else:
        raise ValueError(f"Unsupported accept type: {accept}")


Found the config file in: ./config.json
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


### Sample Data (Synthetic — for demonstration only)

The following cells define sample clinical text documents used in the examples below. When using this notebook with your own data, ensure all inputs are approved synthetic or properly de-identified. Never submit real protected health information (PHI) without explicit authorization and appropriate controls.


In [11]:
docs = ["""ملاحظات سريرية - مريض السكري:
التاريخ: 30 مايو 2023
اسم المريض: فاطمة علي
العنوان: شارع الحرية، مبنى رقم 456، حي السلام، القاهرة
الرمز البريدي: 67890
البلد: مصر
اسم المستشفى: مستشفى الشفاء
اسم الطبيب: د. محمد صلاح
تفاصيل الحالة:
المريضة فاطمة علي، البالغة من العمر 42 عامًا، مصابة بمرض السكري من النوع 2. تشكو من زيادة في العطش والجوع المفرط والتبول المتكرر. تم تشخيصها بمرض السكري بعد فحص شامل وفحوصات مخبرية.
الخطة:
تم وصف دواء فموي لخفض مستوى السكر في الدم.
يجب على المريضة مراجعة الطبيب بانتظام وإجراء اختبارات السكر في الدم بانتظام.
يتعين على المريضة اتباع نظام غذائي صحي ومتوازن، يشمل الحد من استهلاك السكريات والنشويات.
يجب مراقبة ضغط الدم والكولسترول أيضًا ومراعاة التعليمات الطبية المتعلقة بتلك الحالات.
تعليم المريضة بشأن أعراض الارتفاع أو الانخفاض الحاد في مستوى السكر في الدم وكيفية التعامل معها.""",

"""ملاحظات سريرية - مريض الربو:
التاريخ: 30 مايو 2023
اسم المريضة: ليلى حسن
العنوان: شارع المعرفة، مبنى رقم 789، حي الأمانة، جدة
الرمز البريدي: 54321
البلد: المملكة العربية السعودية
اسم المستشفى: مستشفى النور
اسم الطبيب: د. أميرة أحمد
تفاصيل الحالة:
المريضة ليلى حسن، البالغة من العمر 35 عامًا، تعاني من مرض الربو المزمن. تشكو من ضيق التنفس والسعال المتكرر والشهيق الشديد. تم تشخيصها بمرض الربو بناءً على تاريخها الطبي واختبارات وظائف الرئة.
الخطة:
تم وصف مضادات الالتهاب غير الستيرويدية والموسعات القصبية لتحسين التنفس وتقليل التهيج.
يجب على المريضة حمل معها جهاز الاستنشاق في حالة حدوث نوبة ربو حادة.
يتعين على المريضة تجنب التحسس من العوامل المسببة للربو، مثل الدخان والغبار والحيوانات الأليفة.
يجب مراقبة وظائف الرئة بانتظام ومتابعة التعليمات الطبية المتعلقة بمرض الربو.
تعليم المريضة بشأن كيفية استخدام جهاز الاستنشاق بشكل صحيح وتقنيات التنفس الصحيح""",

"""ملاحظات سريرية - مريض الضغط العالي:
التاريخ: 30 مايو 2023
اسم المريض: خالد محمد
العنوان: شارع الشفاء، مبنى رقم 321، حي السعادة، الإسكندرية
الرمز البريدي: 12345
البلد: مصر
اسم المستشفى: مستشفى الرحمة
اسم الطبيب: د. ليلى علي
تفاصيل الحالة:
المريض خالد محمد، البالغ من العمر 60 عامًا، يعاني من ضغط الدم الارتفاعي. يشكو من الصداع المستمر والدوخة والتعب الشديد. تم تشخيصه بارتفاع ضغط الدم بعد فحص شامل وقياسات متكررة.
الخطة:
تم وصف دواءًا لخفض ضغط الدم ومراقبة المستويات بانتظام.
يجب على المريض مراجعة الطبيب بانتظام لتقييم استجابته للعلاج وضبط الجرعات إن لزم الأمر.
تعليم المريض بشأن أهمية اتباع نمط حياة صحي، بما في ذلك تجنب الأطعمة الغنية بالصوديوم وممارسة الرياضة بانتظام.
يتعين على المريض مراقبة الوزن وتجنب التوتر النفسي والتدخين.
ينصح المريض بإجراء فحوصات الكولسترول والسكر في الدم بانتظام.""",

"""ملاحظات سريرية - مريضة الغدة الدرقية:
التاريخ: 30 مايو 2023
اسم المريضة: سارة عبد الرحمن
العنوان: شارع النجاح، مبنى رقم 987، حي الأمل، القاهرة
الرمز البريدي: 67890
البلد: مصر
اسم المستشفى: مستشفى الأمل
اسم الطبيب: د. أحمد مصطفى
تفاصيل الحالة:
المريضة سارة عبد الرحمن، البالغة من العمر 45 عامًا، تعاني من اضطرابات في الغدة الدرقية. تشكو من زيادة الوزن والتعب والاكتئاب. تم تشخيصها بفرط نشاط الغدة الدرقية بعد فحص الدم والتحاليل المخبرية.
الخطة:
تم وصف العلاج الدوائي لتنظيم وظيفة الغدة الدرقية ومراقبة المستويات الهرمونية بانتظام.
يجب على المريضة مراجعة الطبيب بانتظام لتقييم استجابتها للعلاج وضبط الجرعات إن لزم الأمر.
تعليم المريضة بشأن أهمية اتباع نمط حياة صحي، بما في ذلك تناول الغذاء الصحي وممارسة النشاط البدني بانتظام.
ينصح المريضة بإجراء فحوصات الدم الدورية لمتابعة مستويات الهرمونات وضبط العلاج.
تعليم المريضة بشأن الأعراض التي يجب عليها مراقبتها والتقارير الفورية للطبيب، مثل التعب الشديد أو زيادة الوزن المفرط.""",
]


sample_text = """ملاحظات سريرية - مريض الربو:
التاريخ: 30 مايو 2023
اسم المريض: أحمد سليمان
العنوان: شارع السلام، مبنى رقم 555، حي الصفاء، الرياض
الرمز البريدي: 54321
البلد: المملكة العربية السعودية
اسم المستشفى: مستشفى الأمانة
اسم الطبيب: د. ريم الحمد
تفاصيل الحالة:
المريض أحمد سليمان، البالغ من العمر 30 عامًا، يعاني من مرض الربو المزمن. يشكو من ضيق التنفس والسعال المتكرر والشهيق الشديد. تم تشخيصه بمرض الربو بناءً على تاريخه الطبي واختبارات وظائف الرئة.
الخطة:
تم وصف مضادات الالتهاب غير الستيرويدية والموسعات القصبية لتحسين التنفس وتقليل التهيج.
يجب على المريض مراجعة الطبيب بانتظام وتنظيم استخدام جهاز الاستنشاق وفحص وظائف الرئة بانتظام.
ينصح المريض بتجنب التحسس من العوامل المسببة للربو وتجنب التدخين.
يجب على المريض مراقبة الأعراض وتسجيلها والتقارير الفورية للطبيب في حالة حدوث نوبة ربو حادة.
تعليم المريض بشأن تقنيات التنفس الصحيحة وكيفية استخدام جهاز الاستنشاق بشكل صحيح."""

### JSON

#### Example 1: masked (default-policy)

In [21]:
input_json_data = {"text": sample_text}
response_json = invoke_realtime_endpoint(input_json_data, content_type="application/json", accept="application/json")
response_json

{'predictions': 'ملاحظات سريرية - مريض الربو:\nالتاريخ: [تاريخ] [تاريخ]\nاسم المريض: [دكتور]\nالعنوان: شارع السلام، مبنى رقم [الرمز البريدي] حي [الموقع]\nالرمز البريدي: [الرمز البريدي]\nالبلد: [المدينة] [البلد]\nاسم المستشفى: مستشفى الأمانة\nاسم الطبيب: د. [دكتور]\nتفاصيل الحالة:\nالمريض أحمد سليمان، البالغ من العمر [العمر]عامًا، يعاني من مرض الربو المزمن. يشكو من ضيق التنفس والسعال المتكرر والشهيق الشديد. تم تشخيصه بمرض الربو بناءً على تاريخه الطبي واختبارات وظائف الرئة.\nالخطة:\nتم وصف مضادات الالتهاب غير الستيرويدية والموسعات القصبية لتحسين التنفس وتقليل التهيج.\nيجب على المريض مراجعة الطبيب بانتظام وتنظيم استخدام جهاز الاستنشاق وفحص وظائف الرئة بانتظام.\nينصح المريض بتجنب التحسس من العوامل المسببة للربو وتجنب التدخين.\nيجب على المريض مراقبة الأعراض وتسجيلها والتقارير الفورية للطبيب في حالة حدوث نوبة ربو حادة.\nتعليم المريض بشأن تقنيات التنفس الصحيحة وكيفية استخدام جهاز الاستنشاق بشكل صحيح.'}

#### Example 2: obfuscated

In [22]:
input_json_data = {"text": sample_text, "masking_policy": "obfuscated"}
response_json = invoke_realtime_endpoint(input_json_data, content_type="application/json", accept="application/json")
response_json

{'predictions': 'ملاحظات سريرية - مريض الربو:\nالتاريخ: 30 يونيو 2024\nاسم المريض: جاد قفطان\nالعنوان: شارع السلام، مبنى رقم 222، حي شارع المستقبل\nالرمز البريدي: 29630\nالبلد: زغوان الغربية السعودية\nاسم المستشفى: مستشفى الأمانة\nاسم الطبيب: د. رينا سرحان\nتفاصيل الحالة:\nالمريض أحمد سليمان، البالغ من العمر 20عامًا، يعاني من مرض الربو المزمن. يشكو من ضيق التنفس والسعال المتكرر والشهيق الشديد. تم تشخيصه بمرض الربو بناءً على تاريخه الطبي واختبارات وظائف الرئة.\nالخطة:\nتم وصف مضادات الالتهاب غير الستيرويدية والموسعات القصبية لتحسين التنفس وتقليل التهيج.\nيجب على المريض مراجعة الطبيب بانتظام وتنظيم استخدام جهاز الاستنشاق وفحص وظائف الرئة بانتظام.\nينصح المريض بتجنب التحسس من العوامل المسببة للربو وتجنب التدخين.\nيجب على المريض مراقبة الأعراض وتسجيلها والتقارير الفورية للطبيب في حالة حدوث نوبة ربو حادة.\nتعليم المريض بشأن تقنيات التنفس الصحيحة وكيفية استخدام جهاز الاستنشاق بشكل صحيح.'}

### JSON Lines

In [23]:
def create_jsonl(records, masking_policy=None):
    json_records = []

    if isinstance(records, str):
        records = [records]

    for text in records:
        record = {"text": text}

        if masking_policy is not None:
            record["masking_policy"] = masking_policy
        json_records.append(record)

    json_lines = '\n'.join(json.dumps(record, ensure_ascii=False) for record in json_records)
    return json_lines


#### Example 1: masked (default-policy)

In [25]:
input_jsonl_data = create_jsonl(docs, masking_policy="masked")
data = invoke_realtime_endpoint(input_jsonl_data, content_type="application/jsonlines" , accept="application/jsonlines" )
data


[{'predictions': 'ملاحظات سريرية - مريض السكري:\nالتاريخ: [تاريخ] [تاريخ]\nاسم المريض: [الاسم]\nالعنوان: شارع الحرية، مبنى رقم [الرمز البريدي] حي السلام، [المدينة]\nالرمز البريدي: [الرمز البريدي]\nالبلد: [البلد]\nاسم المستشفى: مستشفى الشفاء\nاسم الطبيب: د. [دكتور]\nتفاصيل الحالة:\nالمريضة فاطمة علي، البالغة من العمر [العمر]عامًا، مصابة بمرض السكري من النوع 2. تشكو من زيادة في العطش والجوع المفرط والتبول المتكرر. تم تشخيصها بمرض السكري بعد فحص شامل وفحوصات مخبرية.\nالخطة:\nتم وصف دواء فموي لخفض مستوى السكر في الدم.\nيجب على المريضة مراجعة الطبيب بانتظام وإجراء اختبارات السكر في الدم بانتظام.\nيتعين على المريضة اتباع نظام غذائي صحي ومتوازن، يشمل الحد من استهلاك السكريات والنشويات.\nيجب مراقبة ضغط الدم والكولسترول أيضًا ومراعاة التعليمات الطبية المتعلقة بتلك الحالات.\nتعليم المريضة بشأن أعراض الارتفاع أو الانخفاض الحاد في مستوى السكر في الدم وكيفية التعامل معها.'},
 {'predictions': 'ملاحظات سريرية - مريض الربو:\nالتاريخ: [تاريخ] [تاريخ]\nاسم المريضة: [المريض]\nالعنوان: شارع المعرفة، مبنى 

#### Example 2: obfuscated

In [26]:
input_jsonl_data = create_jsonl(docs, masking_policy="obfuscated")
data = invoke_realtime_endpoint(input_jsonl_data, content_type="application/jsonlines" , accept="application/jsonlines" )
data


[{'predictions': 'ملاحظات سريرية - مريض السكري:\nالتاريخ: 30 يونيو 2024\nاسم المريض: عبدالغني جمالة\nالعنوان: شارع الحرية، مبنى رقم 925، حي السلام، القصر\nالرمز البريدي: 58147\nالبلد: مصر\nاسم المستشفى: مستشفى الشفاء\nاسم الطبيب: د. حمدي نصراوي\nتفاصيل الحالة:\nالمريضة فاطمة علي، البالغة من العمر 53عامًا، مصابة بمرض السكري من النوع 2. تشكو من زيادة في العطش والجوع المفرط والتبول المتكرر. تم تشخيصها بمرض السكري بعد فحص شامل وفحوصات مخبرية.\nالخطة:\nتم وصف دواء فموي لخفض مستوى السكر في الدم.\nيجب على المريضة مراجعة الطبيب بانتظام وإجراء اختبارات السكر في الدم بانتظام.\nيتعين على المريضة اتباع نظام غذائي صحي ومتوازن، يشمل الحد من استهلاك السكريات والنشويات.\nيجب مراقبة ضغط الدم والكولسترول أيضًا ومراعاة التعليمات الطبية المتعلقة بتلك الحالات.\nتعليم المريضة بشأن أعراض الارتفاع أو الانخفاض الحاد في مستوى السكر في الدم وكيفية التعامل معها.'},
 {'predictions': 'ملاحظات سريرية - مريض الربو:\nالتاريخ: 30 يونيو 2024\nاسم المريضة: نوح شقيري\nالعنوان: شارع المعرفة، مبنى رقم 814، حي شارع الحيوانات